In [ ]:
# Import libraries and initialize Earth Engine
import ee
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='qsair-463811')

In [ ]:
# Define field boundary (Sample field in Uganda (approx. 1km x 1km near Tororo))
field_boundary = ee.Geometry.Polygon([
   [34.225, 0.815],  # NW corner
    [34.235, 0.815],  # NE corner
    [34.235, 0.805],  # SE corner
    [34.225, 0.805],  # SW corner
    [34.225, 0.815]   # Closing the polygon (back to NW)
])

In [ ]:
# Get Sentinel-2 imagery and create composite
s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterBounds(field_boundary)
                 .filterDate('2022-06-01', '2022-08-31') # Example peak growing season
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)))

# Create median composite
composite = s2_collection.median()

In [ ]:
# Calculate vegetation indices
# NDVI (Normalized Difference Vegetation Index)
ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
# NDWI (Normalized Difference Water Index)
ndwi = composite.normalizedDifference(['B3', 'B11']).rename('NDWI')
# MNDWI (Modified Normalized Difference Water Index)
mndwi = composite.normalizedDifference(['B3', 'B11']).rename('MNDWI')

In [ ]:
# Create composite with all bands and indices
clustering_bands = [
    'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12',
    'NDVI', 'NDWI', 'MNDWI'
]

# Combine bands into a single image for sampling
clustering_image = composite.addBands([ndvi, ndwi, mndwi])
input_layers_for_clustering = clustering_image.select(clustering_bands)

In [ ]:
# Sample image for training K-means
training_samples = input_layers_for_clustering.sample(
    region=field_boundary,
    scale=10,
    numPixels=4000,
    seed=0,
    tileScale=8
)

# Prepare data for K-means
data = training_samples.getInfo()['features']
properties_list = [f['properties'] for f in data]
df = pd.DataFrame(properties_list)

# Clean data
if 'system:geometry' in df.columns:
    df = df.drop(columns=['system:geometry'])
df = df.dropna()
X = df[clustering_bands]

In [ ]:
# Determine optimal number of clusters (Elbow Method)
inertias = []
silhouette_scores = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)
    if k > 1 and len(np.unique(kmeans.labels_)) > 1:
        silhouette_scores.append(silhouette_score(X, kmeans.labels_))
    else:
        silhouette_scores.append(np.nan)

# Plot Elbow Method
plt.figure(figsize=(10, 5))
plt.plot(K_range, inertias, marker='o')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal Clusters')
plt.grid(True)
plt.show()

In [ ]:
# Plot Silhouette Scores
plt.figure(figsize=(10, 5))
plt.plot(K_range, silhouette_scores, marker='o', color='red')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score for Optimal Clusters')
plt.grid(True)
plt.show()

In [ ]:
# Run final K-means clustering
# Choose optimal K (adjust based on your plots)
optimal_k = 4

# Run final clustering
kmeans_final = KMeans(n_clusters=optimal_k, random_state=42, n_init='auto')
kmeans_final.fit(X)
df['cluster'] = kmeans_final.labels_

In [ ]:
# Visualize clusters
if 'longitude' in df.columns and 'latitude' in df.columns:
    plt.figure(figsize=(10, 8))
    for cluster_label in sorted(df['cluster'].unique()):
        cluster_df = df[df['cluster'] == cluster_label]
        plt.scatter(cluster_df['longitude'], cluster_df['latitude'],
                   label=f'Cluster {cluster_label}', alpha=0.6, s=10)

    plt.xlabel('Longitude')
    plt.ylabel('Latitude')
    plt.title(f'Field Management Zones from K-means Clustering (K={optimal_k})')
    plt.legend(title='Cluster')
    plt.grid(True)
    plt.show()
else:
    print("Warning: 'longitude' and 'latitude' columns not found for direct scatter plotting.")